# 02 - BERTopic refit (MCS=400, BR mayoral-only)

Fresh compact rebuild of the validated pipeline. Run the cells top to bottom on a GPU runtime.
Only two changes vs the prior model: `MCS_FIXED=400` and BR restricted to `valid_mayor_platform == True`. All artifacts write to `data/topics/topic_model_{us,br}/`.

In [2]:
# === 02 - BERTopic refit: MCS=400, BR mayoral-only (US unrestricted) ===
# Recipe UNCHANGED vs the validated model (UMAP nn=60, HDBSCAN ms=25, NcTfidf, KeyBERT;
# outliers -> HDBSnope. stopCAN soft membership; purify: US spread<1% of docs, BR keep spread>20).
# Only two changes: MCS_FIXED 500->400, and BR is restricted to valid_mayor_platform==True.
import torch
assert torch.cuda.is_available(), 'Need a GPU runtime (Runtime > Change runtime type > GPU).'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, random

import os
ROOT = Path(os.getenv('TOPIC2IRT_ROOT', '/content/drive/MyDrive/Papers/transfer_learning/topic2irt'))
RANDOM_STATE = 14605 - 2025 - 4
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

N_COMPONENTS, N_NEIGHBORS, MIN_DIST, METRIC = 10, 60, 0.0, 'euclidean'
MIN_SAMPLES      = 25
USE_OPENAI_LABEL = False
KNN_REASSIGN     = 50
SPREAD_TOPK      = 7
US_SPREAD_FRAC   = 0.01
N_PER_PARTY = 150
N_OOS_DOCS  = 200
N_READ      = 200

MCS_FIXED = 400   # << CHANGED 500 -> 400
def mcs_for(n_chunks):
    if MCS_FIXED is not None:
        return MCS_FIXED
    return int(np.ceil(0.01 * n_chunks / 50.0) * 50)

CORPORA = {
    'us': dict(feat=ROOT/'data/us/campaignview_chunks_sent.feather',
               emb =ROOT/'data/us/emb_minilm.npy',
               lang='english', mode='all', embmodel='all-MiniLM-L6-v2',
               out =ROOT/'data/topics/topic_model_us'),
    'br': dict(feat=ROOT/'data/br/br_manifestos_chunks_sent.feather',
               emb =ROOT/'data/br/emb_minilm_multi.npy',
               lang='multilingual', mode='sample', embmodel='paraphrase-multilingual-MiniLM-L12-v2',
               out =ROOT/'data/topics/topic_model_br'),
}
MAP_BR = ROOT/'data/br/platform_party_map.feather'
for c in CORPORA.values():
    c['out'].mkdir(parents=True, exist_ok=True)
print('config: UMAP(nc=%d, nn=%d, min_dist=%.1f, %s) + HDBSCAN(mcs=%d, ms=%d) | OpenAI_label=%s | kNN=%d'
      % (N_COMPONENTS, N_NEIGHBORS, MIN_DIST, METRIC, MCS_FIXED, MIN_SAMPLES, USE_OPENAI_LABEL, KNN_REASSIGN))
print('US = all chunks (Congress); BR = valid_mayor_platform == True only')

# --- wall time of every step, appended to the pipeline timing register at the end ---
import time as _time
from datetime import datetime
RUN_STARTED = datetime.now()
TIMING = {'us': {}, 'br': {}}
SCALE  = {'us': {}, 'br': {}}
_T0 = {}
def tic():
    _T0['t'] = _time.perf_counter()
def toc(corpus, step):
    TIMING[corpus][step] = _time.perf_counter() - _T0['t']
    print(f"  [time] {corpus} {step}: {TIMING[corpus][step]:,.1f}s")


GPU: NVIDIA A100-SXM4-80GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
config: UMAP(nc=10, nn=60, min_dist=0.0, euclidean) + HDBSCAN(mcs=400, ms=25) | OpenAI_label=False | kNN=50
US = all chunks (Congress); BR = valid_mayor_platform == True only


In [3]:
get_ipython().system('pip install -q bertopic')

import time, gc
import pandas as pd
import scipy.sparse as sp
import pyarrow.feather as feather
from tqdm import tqdm

from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from cuml.cluster.hdbscan import all_points_membership_vectors, membership_vector

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, OpenAI as OpenAIRep

import nltk; nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP = {'english':      stopwords.words('english'),
        'multilingual': list(set(stopwords.words('english') + stopwords.words('portuguese')))}

# --- OpenAI gpt-4o-mini label: WIRED but OFF by default (USE_OPENAI_LABEL). Client built only if ON. ---
OPENAI_PROMPT = (
    "I have a topic that contains the following documents:\n[DOCUMENTS]\n"
    "The topic is described by these keywords: [KEYWORDS]\n\n"
    "Give a label for this topic of AT MOST 3 words, using no stopwords. "
    "Reply with the label only, nothing else.\ntopic: ")

def make_openai_repr():
    import openai
    _client = openai.OpenAI(api_key=(ROOT.parent / 'misc' / 'key.txt').read_text().strip())
    return OpenAIRep(_client, model='gpt-4o-mini', chat=True, prompt=OPENAI_PROMPT,
                     nr_docs=5, doc_length=None, delay_in_seconds=1,
                     generator_kwargs={'max_tokens': 50, 'temperature': 0})


class NcTfidf(BaseEstimator, TransformerMixin):
    """Normalized c-TF-IDF (the weighting used in the tested model). Display-only:
    it sets topic keywords, not the clustering. seed_words/seed_multiplier exist only
    because BERTopic's _c_tf_idf reads them off the ctfidf model."""
    def __init__(self, smooth=1e-6, reduce_frequent_words=False, seed_words=None, seed_multiplier=2.0):
        self.smooth = smooth; self.reduce_frequent_words = reduce_frequent_words
        self.seed_words = seed_words; self.seed_multiplier = seed_multiplier
    def fit(self, X, multiplier=None):
        A = np.asarray(X.todense(), np.float64) if sp.issparse(X) else np.asarray(X, np.float64)
        nt, nterms = A.shape
        idf = np.log(1.0 / ((A > 0).sum(0) / nt + self.smooth))
        self._idf_diag = sp.diags(idf, 0, (nterms, nterms), 'csr', np.float64)
        return self
    def transform(self, X):
        check_is_fitted(self, '_idf_diag')
        X = X.astype(np.float64) if sp.issparse(X) else sp.csr_matrix(X, dtype=np.float64)
        D = sp.diags(1.0 / (np.asarray(X.sum(1)).ravel() + 1e-12), format='csr')
        return sp.csr_matrix((D @ X).multiply(self._idf_diag.diagonal()))
    def fit_transform(self, X, y=None, multiplier=None):
        return self.fit(X, multiplier).transform(X)


def build_topic_model(lang, mcs, embmodel):
    um = UMAP(n_components=N_COMPONENTS, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
              metric=METRIC, random_state=RANDOM_STATE)
    hd = HDBSCAN(min_cluster_size=mcs, min_samples=MIN_SAMPLES,
                 metric=METRIC, prediction_data=True)
    cv = CountVectorizer(ngram_range=(1, 1), stop_words=STOP[lang],
                         min_df=2, max_df=0.95, max_features=50_000)   # UNIGRAM keywords
    reps = {'KeyBERT': KeyBERTInspired()}                              # 2nd keyword form (embedding-guided)
    if USE_OPENAI_LABEL:
        reps['OpenAI'] = make_openai_repr()                           # optional <=3-word label aspect
    print('  build: mcs=%d nn=%d unigram | reps=%s | embed=%s' % (mcs, N_NEIGHBORS, list(reps), embmodel))
    return BERTopic(embedding_model=SentenceTransformer(embmodel), umap_model=um, hdbscan_model=hd,
                    vectorizer_model=cv, ctfidf_model=NcTfidf(),
                    representation_model=reps,
                    calculate_probabilities=False, verbose=True)


def topic_desc(tm, t):
    """The gpt-4o-mini <=3-word label for a topic (the 'OpenAI' aspect); '' if OpenAI is off."""
    if int(t) == -1:
        return '(no topic)'
    asp = tm.topic_aspects_.get('OpenAI', {}).get(int(t))
    return asp[0][0] if asp else ''


def kw_ctfidf(tm, t):
    """Normalized c-TF-IDF keywords (the MAIN representation)."""
    return ', '.join(w for w, _ in tm.get_topic(int(t))[:12]) if int(t) != -1 else ''


def kw_keybert(tm, t):
    """KeyBERT (embedding-guided) keywords (the 2nd representation aspect)."""
    asp = tm.topic_aspects_.get('KeyBERT', {}).get(int(t))
    return ', '.join(w for w, _ in asp[:12]) if asp else ''


def hdbscan_L2T(tm, topics):
    """map raw HDBSCAN cluster id (membership-vector column) -> BERTopic topic id."""
    raw = np.asarray(tm.hdbscan_model.labels_)
    return {int(l): int(t) for l, t in zip(raw, topics) if l != -1}


def assign_train_outliers(tm, topics):
    """Reassign every -1 to its top HDBSCAN soft-membership cluster. Returns (final_topics, outlier_mask)."""
    L2T  = hdbscan_L2T(tm, topics)
    apmv = np.asarray(all_points_membership_vectors(tm.hdbscan_model))   # (n_train, n_clusters)
    L    = apmv.argmax(1)
    final = np.asarray(topics).copy()
    out   = final == -1
    final[out] = np.array([L2T.get(int(l), -1) for l in L])[out]
    return final, out


def assign_oos(tm, L2T, emb_oos, batch=100_000):
    """Project unseen embeddings into the fitted UMAP, assign by HDBSCAN soft membership argmax."""
    out = np.empty(len(emb_oos), np.int64)
    for s in tqdm(range(0, len(emb_oos), batch), unit='blk', desc='project'):
        coords = np.asarray(tm.umap_model.transform(emb_oos[s:s+batch]))
        L = np.asarray(membership_vector(tm.hdbscan_model, coords)).argmax(1)
        out[s:s+batch] = [L2T.get(int(l), -1) for l in L]
    return out


def export_for_reading(path, idx, texts, topic_ids, tm):
    rows = []
    for i in idx:
        t  = int(topic_ids[i])
        kw = kw_ctfidf(tm, t) if t != -1 else '(no topic)'
        rows.append(dict(row=int(i), topic=t, description=topic_desc(tm, t), keywords=kw,
                         chunk_text=str(texts[i])[:300], correct=''))
    pd.DataFrame(rows).to_csv(path, index=False)
    print('wrote', path.name, '(', len(rows), 'rows for manual reading )')


def save_codebook(tm, final_topics, out_dir):
    """Codebook with BOTH keyword forms: normalized c-TF-IDF (keywords) and KeyBERT (keywords_keybert)."""
    counts = pd.Series(final_topics).value_counts().sort_index()
    rows = [dict(topic=int(t), description=topic_desc(tm, int(t)),
                 keywords=kw_ctfidf(tm, int(t)), keywords_keybert=kw_keybert(tm, int(t)),
                 n_chunks=int(counts.get(t, 0)))
            for t in counts.index]
    cb = pd.DataFrame(rows)
    cb.to_csv(out_dir / 'topics_codebook.csv', index=False)
    print('codebook:', len(cb), 'topics ->', (out_dir / 'topics_codebook.csv').name)
    return cb

print('cuML + BERTopic + KeyBERT (+optional OpenAI) helpers ready')

# --- PURIFICATION helpers: drop "almost mono-doc" topics, kNN-reassign their chunks ---
# kw_doc_spread(topic) = median, over the topic's top-SPREAD_TOPK c-TF-IDF keywords, of the number of
#   DISTINCT docs that use that keyword corpus-wide. Low spread => the topic's vocabulary lives in a
#   handful of documents (an idiosyncratic near-mono-doc cluster), so we mask it and reassign its chunks.
# NOTE: text is clean UTF-8 on Colab (Linux) -> keywords match verbatim; NO ascii/accent stripping.

def topic_keywords(tm, topk=SPREAD_TOPK):
    "top-k normalized c-TF-IDF keywords per (non-outlier) topic."
    out = {}
    for t in tm.get_topics():
        t = int(t)
        if t < 0:
            continue
        out[t] = [w for w, _ in tm.get_topic(t)[:topk] if w]
    return out

def kw_doc_spread(kw_by_topic, texts, doc_ids, topk=SPREAD_TOPK):
    "dict topic -> median distinct-doc count over its top-k keywords (one vectorized pass)."
    vocab = sorted({w for ks in kw_by_topic.values() for w in ks[:topk] if w})
    cv = CountVectorizer(vocabulary=vocab, binary=True, lowercase=True, ngram_range=(1, 1))
    Xc = cv.transform(texts)                                    # (n_chunk x V) binary
    codes, _ = pd.factorize(np.asarray(doc_ids))
    Dinc = sp.csr_matrix((np.ones(len(codes), np.float32), (codes, np.arange(len(codes)))))
    B = Dinc @ Xc; B.data = (B.data > 0).astype(np.float32)     # (n_doc x V) doc-has-keyword
    docfreq = np.asarray(B.sum(0)).ravel().astype(int)
    vi = cv.vocabulary_
    return {t: (float(np.median([docfreq[vi[k]] for k in ks[:topk] if k in vi]))
                if any(k in vi for k in ks[:topk]) else np.nan)
            for t, ks in kw_by_topic.items()}

def knn_reassign(emb, topics, masked_set, k=KNN_REASSIGN, max_sim_bytes=2.5e9):
    "reassign every masked-topic chunk to the majority topic among its k nearest valid-topic chunks (cosine, GPU)."
    topics = np.asarray(topics).copy()
    masked_set = {int(t) for t in masked_set}
    is_masked = np.fromiter((int(t) in masked_set for t in topics), bool, len(topics))
    valid = (~is_masked) & (topics >= 0)
    q_idx = np.where(is_masked)[0]
    if len(q_idx) == 0:
        print('  kNN: nothing masked -> no reassignment'); return topics, is_masked, q_idx
    dev = 'cuda'
    Xv = torch.from_numpy(np.ascontiguousarray(emb[valid], dtype=np.float32)).to(dev)  # memmap fancy-index -> RAM
    torch.nn.functional.normalize(Xv, dim=1, out=Xv)
    Pt = torch.from_numpy(topics[valid].astype(np.int64)).to(dev)
    nvalid = Xv.shape[0]
    qb = max(1, min(8192, int(max_sim_bytes / (nvalid * 4))))
    new = topics.copy()
    for s in tqdm(range(0, len(q_idx), qb), unit='blk', desc=f'kNN reassign (k={k})'):
        qi = q_idx[s:s+qb]
        Q = torch.from_numpy(np.ascontiguousarray(emb[qi], dtype=np.float32)).to(dev)
        torch.nn.functional.normalize(Q, dim=1, out=Q)
        nn = (Q @ Xv.T).topk(k, dim=1).indices                  # (qb, k) nearest valid neighbours
        new[qi] = torch.mode(Pt[nn], dim=1).values.cpu().numpy()
        del Q, nn
    del Xv, Pt; gc.collect(); torch.cuda.empty_cache()
    print(f'  kNN(k={k}): reassigned {len(q_idx):,} masked chunks | pool {nvalid:,} valid | qbatch {qb}')
    return new, is_masked, q_idx

def purify(tm, emb, texts, doc_ids, topics, thresh, k=KNN_REASSIGN):
    "spread -> mask topics < thresh -> kNN-reassign their chunks. `emb` is the chunk embedding matrix/memmap."
    kw_by_topic = topic_keywords(tm)
    spread = kw_doc_spread(kw_by_topic, texts, doc_ids)
    masked = sorted([t for t, s in spread.items() if t >= 0 and (not np.isnan(s)) and s < thresh])
    print(f'purify: thresh={thresh} | {len(masked)}/{len(spread)} topics masked (kw_doc_spread<{thresh})')
    new_topics, is_masked, q_idx = knn_reassign(emb, topics, set(masked), k=k)
    return dict(spread=spread, masked=set(masked), new_topics=new_topics,
                is_masked=is_masked, q_idx=q_idx, thresh=thresh, kw_by_topic=kw_by_topic)

def save_codebook_spread(tm, final_topics, spread, out_dir):
    "codebook over PURIFIED topics: c-TF-IDF + KeyBERT keywords + kw_doc_spread + n_chunks."
    counts = pd.Series(final_topics).value_counts().sort_index()
    rows = [dict(topic=int(t), description=topic_desc(tm, int(t)),
                 keywords=kw_ctfidf(tm, int(t)), keywords_keybert=kw_keybert(tm, int(t)),
                 kw_doc_spread=spread.get(int(t), np.nan), n_chunks=int(counts.get(t, 0)))
            for t in counts.index]
    cb = pd.DataFrame(rows); cb.to_csv(out_dir / 'topics_codebook.csv', index=False)
    print('codebook:', len(cb), 'purified topics ->', (out_dir / 'topics_codebook.csv').name)
    return cb

def save_spread_table(spread, masked_set, orig_topics, thresh, out_dir):
    ot = np.asarray(orig_topics)
    cnt = pd.Series(ot[ot >= 0]).value_counts()
    df = pd.DataFrame({'topic': list(spread), 'kw_doc_spread': list(spread.values())})
    df['n_chunks_orig'] = df.topic.map(cnt).fillna(0).astype(int)
    df['masked'] = df.topic.isin(masked_set); df['threshold'] = thresh
    df = df.sort_values('kw_doc_spread')
    df.to_csv(out_dir / 'topics_kw_doc_spread.csv', index=False)
    print('spread table ->', (out_dir / 'topics_kw_doc_spread.csv').name, '| masked', int(df.masked.sum()))
    return df

print('purification helpers ready (kw_doc_spread + knn_reassign + purify)')

# BEFORE/AFTER export helpers: recompute c-TF-IDF keywords (strict unigram vocab) on BOTH the pre-purify
# labels (topic_orig) and the post-purify labels (topic), from a 100-chunk/topic sample. Fast + corpus-wide.
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

def sampled_ctfidf_keywords(ft, labelcol, lang, k=100, topn=12):
    d = ft[ft[labelcol] >= 0]
    parts = [g.sample(min(k, len(g)), random_state=0) for _, g in d.groupby(labelcol)]
    samp = pd.concat(parts)
    cv = CountVectorizer(ngram_range=(1, 1), stop_words=STOP[lang], min_df=5, max_df=0.5,
                         token_pattern=r"(?u)\b[^\W\d_]{3,}\b", max_features=50_000)
    X = cv.fit_transform(samp['chunk_text']); terms = np.array(cv.get_feature_names_out())
    tks = np.sort(samp[labelcol].unique()); tj = {t: j for j, t in enumerate(tks)}
    R = sp.csr_matrix((np.ones(len(samp)), ([tj[t] for t in samp[labelcol]], np.arange(len(samp)))))
    W = np.asarray(NcTfidf().fit_transform(R @ X).todense())
    return {int(tks[j]): ", ".join(terms[np.argsort(-W[j])[:topn]]) for j in range(len(tks))}

def export_before_after(out_dir, lang):
    ft = feather.read_table(out_dir / 'chunk_topics.feather').to_pandas()
    kw_before = sampled_ctfidf_keywords(ft, 'topic_orig', lang)     # pre-purify clusters (all topics)
    kw_after  = sampled_ctfidf_keywords(ft, 'topic', lang)          # post-purify (survivors, reassigned)
    n_before  = ft[ft.topic_orig >= 0].topic_orig.value_counts()
    n_after   = ft[ft.topic >= 0].topic.value_counts()
    spf = out_dir / 'topics_kw_doc_spread.csv'
    spt = pd.read_csv(spf).set_index('topic') if spf.exists() else pd.DataFrame()
    allt = sorted(set(kw_before) | set(kw_after))
    rows = [dict(topic=t,
                 n_chunks_before=int(n_before.get(t, 0)), n_chunks_after=int(n_after.get(t, 0)),
                 masked=bool(spt['masked'].get(t, False)) if len(spt) else False,
                 kw_doc_spread=float(spt['kw_doc_spread'].get(t, np.nan)) if len(spt) else np.nan,
                 keywords_before=kw_before.get(t, ''), keywords_after=kw_after.get(t, ''))
            for t in allt]
    out = pd.DataFrame(rows)
    out.to_excel(out_dir / 'topic_info_before_after.xlsx', index=False)
    print(f'saved {out_dir.name}/topic_info_before_after.xlsx | {len(out)} topics '
          f'| masked {int(out.masked.sum())} | cols {list(out.columns)}')
    return out

print('before/after export helpers ready')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.6 MB/s eta 0:00:00
cuML + BERTopic + KeyBERT (+optional OpenAI) helpers ready
purification helpers ready (kw_doc_spread + knn_reassign + purify)
before/after export helpers ready


In [4]:
print('===== US PIPELINE =====')
# US — LOAD data (chunks + embeddings), aligned to feather chunk_id order via *_chunk_ids.npy (defensive;
# US matrix is already in-order, so this is a no-op for US — but it guarantees correctness either way).
tic()
cfg_us  = CORPORA['us']
df_us   = feather.read_table(cfg_us['feat'],
              columns=['chunk_id', 'doc_id', 'party_label', 'chunk_text']).to_pandas()
docs_us = df_us['chunk_text'].tolist()
_ids = np.load(str(cfg_us['emb']).replace('.npy', '_chunk_ids.npy'), allow_pickle=True)
_pos = {str(c): i for i, c in enumerate(_ids)}
_rowidx = df_us['chunk_id'].astype(str).map(_pos)
assert _rowidx.notna().all(), 'some feather chunk_id missing from embedding ids'
emb_us  = np.load(cfg_us['emb'])[_rowidx.astype(np.int64).to_numpy()].astype('float32')
assert len(df_us) == len(emb_us)
print(f"US loaded + ALIGNED by chunk_id: {len(df_us):,} chunks | emb {emb_us.shape}")
SCALE['us'] = {'n_docs': int(df_us['doc_id'].nunique()), 'n_chunks': int(len(df_us))}
toc('us', '1 load+align')

# US — TRAIN BERTopic on all chunks (fit ONLY; does not save).
#      To reuse a saved model, run the LOAD-MODEL cell instead — never refit.
t = time.time()
mcs_us = mcs_for(len(docs_us))                      # 1% of all US chunks, up to x50
tm_us = build_topic_model(cfg_us['lang'], mcs_us, cfg_us['embmodel'])
topics_us, _ = tm_us.fit_transform(docs_us, embeddings=emb_us)
topics_us = np.asarray(topics_us)
print(f"fit {time.time()-t:.0f}s | mcs={mcs_us} | {int((np.unique(topics_us)!=-1).sum())} topics | "
      f"raw outliers {100*(topics_us==-1).mean():.1f}%")
TIMING['us']['2 fit'] = time.time() - t

# US — SAVE the fitted model.
# The OpenAI representation holds a live httpx client (unpicklable _thread.RLock).
# Its labels are already stored in tm_us.topic_aspects_, so drop the live client
# before pickling — labels + the cuML UMAP/HDBSCAN (needed for projection) are kept.
tic()
tm_us.representation_model = None
tm_us.save(str(cfg_us['out'] / 'bertopic_model'), serialization='pickle')
print('saved', (cfg_us['out'] / 'bertopic_model').name)
toc('us', '3 save model')

# US — FINALIZE: assign training outliers via HDBSCAN membership, write labelled chunks + codebook
tic()
final_us, out_us = assign_train_outliers(tm_us, topics_us)
print(f"outliers after membership argmax: {100*(final_us==-1).mean():.2f}% (reassigned {out_us.sum():,})")

df_us['topic'] = final_us
df_us['was_outlier'] = out_us
feather.write_feather(df_us, cfg_us['out'] / 'chunk_topics.feather')
save_codebook(tm_us, final_us, cfg_us['out'])
print('saved chunk_topics.feather + codebook')
toc('us', '4 outlier assign + write')

# US — PURIFY: mask near-mono-doc topics (kw_doc_spread < 1% of docs) + kNN=50 reassign their chunks.
# Runs on the finalized assignment (final_us, post outlier-membership). Overwrites chunk_topics.feather.
tic()
n_docs_us = df_us['doc_id'].nunique()
THRESH_US = max(1, int(np.ceil(US_SPREAD_FRAC * n_docs_us)))
print(f'US docs {n_docs_us:,} | spread threshold = {THRESH_US} (= ceil {US_SPREAD_FRAC:.0%} of docs)')

res_us  = purify(tm_us, emb_us, df_us['chunk_text'], df_us['doc_id'].to_numpy(), final_us, THRESH_US)
pure_us = res_us['new_topics']
print(f"  topics {len(set(final_us[final_us>=0]))} -> {len(set(pure_us[pure_us>=0]))} after purge "
      f"| reassigned {res_us['q_idx'].size:,} | unassigned {100*(pure_us==-1).mean():.2f}%")

df_us['topic_orig']  = final_us
df_us['topic']       = pure_us
df_us['was_outlier'] = out_us
feather.write_feather(df_us, cfg_us['out'] / 'chunk_topics.feather')
save_spread_table(res_us['spread'], res_us['masked'], final_us, THRESH_US, cfg_us['out'])
save_codebook_spread(tm_us, pure_us, res_us['spread'], cfg_us['out'])
print('US purified -> chunk_topics.feather (topic=purified, topic_orig=pre-purge) + codebook + spread table')
toc('us', '5 purify + kNN')

tic()
# US — export a manual-reading sample of ASSIGNED OUTLIERS (accuracy check)
rng = np.random.default_rng(RANDOM_STATE)
out_pos = np.where(out_us)[0]
idx = rng.choice(out_pos, size=min(N_READ, len(out_pos)), replace=False)
export_for_reading(cfg_us['out'] / 'read_assigned_outliers.csv', idx, docs_us, final_us, tm_us)

# US — EXPORT topic documentation to Excel with BOTH pre- and post-purification info in one file:
#   get_topic_info() (Representation=c-TF-IDF, KeyBERT, Representative_Docs) + kw_doc_spread + masked
#   + n_chunks_orig (full-corpus PRE-purify) + n_chunks_purified (full-corpus POST-purify; 0 if masked).
def _flat(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return ' | '.join(_flat(e) for e in v if str(e).strip() != '')
    return v
info = tm_us.get_topic_info().copy()
for c in info.columns:
    if info[c].map(lambda v: isinstance(v, (list, tuple, np.ndarray))).any():
        info[c] = info[c].map(_flat)

# full-corpus pre/post counts from the saved chunk table (topic_orig = pre-purify, topic = post-purify)
ft = feather.read_table(cfg_us['out'] / 'chunk_topics.feather').to_pandas()
has_orig = 'topic_orig' in ft.columns
oc = (ft['topic_orig'] if has_orig else ft['topic']).value_counts()
pc = ft['topic'].value_counts()
info['n_chunks_orig']     = info['Topic'].map(oc).fillna(0).astype(int)   # PRE-purification (full corpus)
info['n_chunks_purified'] = info['Topic'].map(pc).fillna(0).astype(int)   # POST-purification (0 => masked-away)

sp_path = cfg_us['out'] / 'topics_kw_doc_spread.csv'
if sp_path.exists():
    spdf = pd.read_csv(sp_path)[['topic', 'kw_doc_spread', 'masked']]      # spdf, NOT sp (sp = scipy.sparse!)
    info = info.merge(spdf, left_on='Topic', right_on='topic', how='left').drop(columns='topic')
else:
    print('  (topics_kw_doc_spread.csv not found — run PURIFY/SPREAD first for kw_doc_spread)')
info.to_excel(cfg_us['out'] / 'topic_info.xlsx', index=False)
print('saved topic_info.xlsx |', len(info), 'topics | cols:', list(info.columns))

# US — VERIFY the exported Excel actually has keywords + OpenAI label + exemplar docs
chk = pd.read_excel(cfg_us['out'] / 'topic_info.xlsx')
print('columns:', list(chk.columns))
need = ['Topic', 'Count', 'Representation', 'OpenAI', 'Representative_Docs']
print('MISSING:', [c for c in need if c not in chk.columns])
for c in [x for x in ['Representation', 'OpenAI', 'Representative_Docs'] if x in chk.columns]:
    empty = chk[c].isna() | (chk[c].astype(str).str.strip().isin(['', 'nan']))
    print(f'  {c}: {int(empty.sum())}/{len(chk)} empty')
with pd.option_context('display.max_colwidth', 100, 'display.width', 200):
    print(chk.head(3).to_string())
toc('us', '6 exports')


===== US PIPELINE =====
US loaded + ALIGNED by chunk_id: 439,282 chunks | emb (439282, 384)
  [time] us 1 load+align: 13.6s
  build: mcs=400 nn=60 unigram | reps=['KeyBERT'] | embed=all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-08-13 02:55:31,150 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-13 02:56:01,410 - BERTopic - Dimensionality - Completed ✓
2026-08-13 02:56:01,439 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-13 02:56:06,639 - BERTopic - Cluster - Completed ✓
2026-08-13 02:56:06,722 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-13 02:56:15,650 - BERTopic - Representation - Completed ✓
2026-08-13 02:56:16,629 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


fit 53s | mcs=400 | 101 topics | raw outliers 39.2%
saved bertopic_model
  [time] us 3 save model: 27.6s
outliers after membership argmax: 0.00% (reassigned 171,981)
codebook: 101 topics -> topics_codebook.csv
saved chunk_topics.feather + codebook
  [time] us 4 outlier assign + write: 8.1s
US docs 4,507 | spread threshold = 46 (= ceil 1% of docs)
purify: thresh=46 | 6/101 topics masked (kw_doc_spread<46)


kNN reassign (k=50): 100%|██████████| 12/12 [00:00<00:00, 23.17blk/s]


  kNN(k=50): reassigned 16,303 masked chunks | pool 422,979 valid | qbatch 1477
  topics 101 -> 95 after purge | reassigned 16,303 | unassigned 0.00%
spread table -> topics_kw_doc_spread.csv | masked 6
codebook: 95 purified topics -> topics_codebook.csv
US purified -> chunk_topics.feather (topic=purified, topic_orig=pre-purge) + codebook + spread table
  [time] us 5 purify + kNN: 33.0s
wrote read_assigned_outliers.csv ( 200 rows for manual reading )
saved topic_info.xlsx | 102 topics | cols: ['Topic', 'Count', 'Name', 'Representation', 'KeyBERT', 'Representative_Docs', 'n_chunks_orig', 'n_chunks_purified', 'kw_doc_spread', 'masked']
columns: ['Topic', 'Count', 'Name', 'Representation', 'KeyBERT', 'Representative_Docs', 'n_chunks_orig', 'n_chunks_purified', 'kw_doc_spread', 'masked']
MISSING: ['OpenAI']
  Representation: 0/102 empty
  Representative_Docs: 0/102 empty
   Topic   Count                                       Name                                                              

In [5]:
print('===== BR: load + mayoral filter + sample =====')
# BR — LOAD data (chunks + embeddings). Embeddings were saved LENGTH-SORTED, so we MUST reorder them
# to the feather's chunk_id order via the *_chunk_ids.npy key. (A length check alone is NOT alignment!)
tic()
cfg_br = CORPORA['br']
br = feather.read_table(cfg_br['feat'],
        columns=['chunk_id', 'doc_id', 'party_label', 'chunk_text']).to_pandas()
_ids = np.load(str(cfg_br['emb']).replace('.npy', '_chunk_ids.npy'), allow_pickle=True)
_pos = {str(c): i for i, c in enumerate(_ids)}
_rowidx = br['chunk_id'].astype(str).map(_pos)
assert _rowidx.notna().all(), 'some feather chunk_id missing from embedding ids'
_rowidx = _rowidx.astype(np.int64).to_numpy()
emb_br = np.asarray(np.load(cfg_br['emb'], mmap_mode='r')[_rowidx])   # reorder rows -> aligned to br
assert len(br) == len(emb_br) == _rowidx.size
print(f"BR loaded + ALIGNED by chunk_id: {len(br):,} chunks over {br['doc_id'].nunique():,} docs, "
      f"{br['party_label'].nunique()} parties | emb {emb_br.shape}")

# --- MAYORAL RESTRICTION (BR only): keep only platforms with valid_mayor_platform == True ---
# doc_id == platform_id in the BR chunk table; US has no such flag and is left unrestricted.
_pm = pd.read_feather(MAP_BR, columns=['platform_id', 'valid_mayor_platform'])
_mayor = set(_pm.loc[_pm['valid_mayor_platform'] == True, 'platform_id'].astype(str))
_keep = br['doc_id'].astype(str).isin(_mayor).to_numpy()
print('BR mayoral filter: %d docs / %d chunks  ->  %d mayoral docs / %d chunks (dropped %d docs)'
      % (br['doc_id'].nunique(), len(br),
         br.loc[_keep, 'doc_id'].nunique(), int(_keep.sum()),
         br['doc_id'].nunique() - br.loc[_keep, 'doc_id'].nunique()))
br = br.loc[_keep].reset_index(drop=True)
emb_br = emb_br[_keep]
assert len(br) == len(emb_br)

# BR — SAMPLE: 150 docs/party for TRAIN; 200 other docs for OUT-OF-SAMPLE (deterministic)
rng = np.random.default_rng(RANDOM_STATE)
doc_party = br.drop_duplicates('doc_id')[['doc_id', 'party_label']]

train_docs = []
for p, g in doc_party.groupby('party_label'):
    ids = g['doc_id'].values
    train_docs.extend(rng.choice(ids, size=min(N_PER_PARTY, len(ids)), replace=False))
train_docs = set(train_docs)

rest = doc_party.loc[~doc_party['doc_id'].isin(train_docs), 'doc_id'].values
oos_docs = set(rng.choice(rest, size=min(N_OOS_DOCS, len(rest)), replace=False))

train_mask = br['doc_id'].isin(train_docs).values
oos_mask   = br['doc_id'].isin(oos_docs).values
docs_tr = br.loc[train_mask, 'chunk_text'].tolist()
print(f"train: {len(train_docs):,} docs / {train_mask.sum():,} chunks | "
      f"OOS: {len(oos_docs):,} docs / {oos_mask.sum():,} chunks")
SCALE['br'] = {'n_docs': int(br['doc_id'].nunique()), 'n_chunks': int(len(br))}
toc('br', '1 load+align+filter+sample')


===== BR: load + mayoral filter + sample =====
BR loaded + ALIGNED by chunk_id: 3,173,387 chunks over 17,385 docs, 36 parties | emb (3173387, 384)
BR mayoral filter: 17385 docs / 3173387 chunks  ->  16830 mayoral docs / 3081612 chunks (dropped 555 docs)
train: 4,067 docs / 767,699 chunks | OOS: 200 docs / 33,630 chunks
  [time] br 1 load+align+filter+sample: 76.3s


In [6]:
print('===== BR: train + finalize + re-represent + label-all =====')
# BR — TRAIN BERTopic on the sampled chunks (fit ONLY; does not save).
#      To reuse a saved model, run the LOAD-MODEL cell instead — never refit.
Xtr = np.asarray(emb_br[train_mask]).astype('float32')
print('Xtr', Xtr.shape)
t = time.time()
mcs_br = mcs_for(int(train_mask.sum()))             # 1% of BR sampled chunks, up to x50
tm_br = build_topic_model(cfg_br['lang'], mcs_br, cfg_br['embmodel'])
topics_tr, _ = tm_br.fit_transform(docs_tr, embeddings=Xtr)
topics_tr = np.asarray(topics_tr)
print(f"fit {time.time()-t:.0f}s | mcs={mcs_br} | {int((np.unique(topics_tr)!=-1).sum())} topics | "
      f"raw outliers {100*(topics_tr==-1).mean():.1f}%")
TIMING['br']['2 fit'] = time.time() - t

# BR — SAVE the fitted model.
# The OpenAI representation holds a live httpx client (unpicklable _thread.RLock).
# Its labels are already stored in tm_br.topic_aspects_, so drop the live client
# before pickling — labels + the cuML UMAP/HDBSCAN (needed for projection) are kept.
tic()
tm_br.representation_model = None
tm_br.save(str(cfg_br['out'] / 'bertopic_model'), serialization='pickle')
print('saved', (cfg_br['out'] / 'bertopic_model').name)
toc('br', '3 save model')

# BR — FINALIZE: assign training outliers via HDBSCAN membership, build L2T map, save train labels + codebook
tic()
final_tr, out_tr = assign_train_outliers(tm_br, topics_tr)
L2T_br = hdbscan_L2T(tm_br, topics_tr)
print(f"outliers after membership argmax: {100*(final_tr==-1).mean():.2f}% (reassigned {out_tr.sum():,})")
np.save(cfg_br['out'] / 'train_topics.npy', final_tr)
save_codebook(tm_br, final_tr, cfg_br['out'])
print('saved train_topics.npy + codebook')
toc('br', '4 outlier assign + write')

# BR — RE-REPRESENT keywords on the EXISTING clusters with a strict unigram vocab (NO refit / NO re-cluster).
# Fixes the "trash keyword" problem: min_df=25 + max_df=0.5 + no-digit token pattern removes hyperlocal
# names, numbers, and OCR glue-tokens. Clusters (chunk->topic) are UNCHANGED; only labels are recomputed.
tic()
_cv = CountVectorizer(ngram_range=(1, 1), stop_words=STOP[cfg_br['lang']],
                      min_df=25, max_df=0.5,
                      token_pattern=r"(?u)\b[^\W\d_]{3,}\b",      # >=3 letters, NO digits
                      max_features=50_000)
tm_br.update_topics(docs_tr, topics=final_tr.tolist(), vectorizer_model=_cv,
                    ctfidf_model=NcTfidf(), representation_model={'KeyBERT': KeyBERTInspired()})
print('BR re-represented (unigram, min_df=25, max_df=0.5, no digits). sample:')
for t in [0, 1, 2, 9, 16, 22, 32]:
    print(f'  T{t}: {", ".join(w for w, _ in tm_br.get_topic(t)[:10])}')

toc('br', '5 re-represent')
tic()
# BR — label ALL chunks (block-checkpointed so a reset resumes). Needs tm_br + L2T_br +
# train_mask + final_tr live; if the kernel reset, re-run BR load + sample + LOAD-MODEL + FINALIZE first.
import shutil
lab_path = cfg_br['out'] / 'chunk_topics.feather'
pdir = cfg_br['out'] / 'label_parts'; pdir.mkdir(exist_ok=True)
B = 100_000

rest_pos = np.where(~train_mask)[0]
for s in tqdm(range(0, len(rest_pos), B), unit='blk', desc='label rest'):
    bf = pdir / f'{s:09d}.npy'
    if bf.exists():                                   # resume: block already projected
        continue
    pos = rest_pos[s:s+B]
    coords = np.asarray(tm_br.umap_model.transform(np.asarray(emb_br[pos]).astype('float32')))
    L = np.asarray(membership_vector(tm_br.hdbscan_model, coords)).argmax(1)
    np.save(bf, np.array([L2T_br.get(int(l), -1) for l in L], np.int64))

all_topics = np.full(len(br), -2, np.int64)
all_topics[train_mask] = final_tr
for s in range(0, len(rest_pos), B):
    all_topics[rest_pos[s:s+B]] = np.load(pdir / f'{s:09d}.npy')
assert (all_topics != -2).all()

br['topic'] = all_topics
br['in_train'] = train_mask
# --- guarantee party info in the saved topic table (robust to text-source changes) ---
if 'platform_id' not in br.columns:
    br['platform_id'] = br['doc_id']
if ('party_label' not in br.columns) or br['party_label'].isna().all():
    _pm = pd.read_feather(ROOT / 'data/br/platform_party_map.feather')[['platform_id', 'party']]
    _pm['platform_id'] = _pm['platform_id'].astype(str)
    br['party_label'] = br['platform_id'].astype(str).map(dict(zip(_pm['platform_id'], _pm['party'])))
feather.write_feather(br, lab_path)
shutil.rmtree(pdir)
print('saved', lab_path.name, '|', f"{100*(all_topics==-1).mean():.2f}% unassigned over {len(br):,} chunks")
toc('br', '6 label all chunks')


===== BR: train + finalize + re-represent + label-all =====
Xtr (767699, 384)
  build: mcs=400 nn=60 unigram | reps=['KeyBERT'] | embed=paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

fit 151s | mcs=400 | 168 topics | raw outliers 50.3%
saved bertopic_model
  [time] br 3 save model: 71.9s
outliers after membership argmax: 0.00% (reassigned 385,924)


2026-08-13 03:02:59,355 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


codebook: 168 topics -> topics_codebook.csv
saved train_topics.npy + codebook
  [time] br 4 outlier assign + write: 7.2s
BR re-represented (unigram, min_df=25, max_df=0.5, no digits). sample:
  T0: artistas, música, festival, turístico, festas, teatro, festivais, turísticos, manifestações, dança
  T1: futebol, campeonatos, atletas, esportivos, esportiva, competições, jogos, quadras, campeonato, esportivo
  T2: câmeras, polícia, militar, polícias, policiais, criminalidade, policia, policial, guardas, policiamento
  T9: pluviais, nascentes, rios, potável, esgoto, barragens, drenagem, hídricos, poços, artesianos
  T16: deslocam, universitários, universitário, motivar, estudam, talentos, bebidas, monitores, armas, residem
  T22: biblioteca, bibliotecas, leitura, acervo, livros, livro, dotada, literatura, preparatório, enem
  T32: estradas, vicinais, escoamento, propriedades, agrícola, bueiros, tráfego, pavimentação, escoar, estrada
  [time] br 5 re-represent: 15.5s


label rest: 100%|██████████| 24/24 [05:47<00:00, 14.46s/blk]


saved chunk_topics.feather | 0.00% unassigned over 3,081,612 chunks
  [time] br 6 label all chunks: 355.4s


In [7]:
print('===== BR: purify + reads + topic_info =====')
# BR - PURIFY: mask near-mono-doc topics + kNN=50 reassign their chunks. Runs on the FULL labelled corpus.
# Needs br (br['topic'] = raw label-all assignment) + emb_br + tm_br live.
# keep kw_doc_spread > 20; mask <= 20  (THRESH=20.5 on the 0.5 median grid - see BR SPREAD cell).
tic()
THRESH_BR = 20.5
print(f'BR spread threshold = {THRESH_BR} (keep kw_doc_spread > 20) | total docs {br.doc_id.nunique():,}')

orig_br = br['topic'].to_numpy()
res_br  = purify(tm_br, emb_br, br['chunk_text'], br['doc_id'].to_numpy(), orig_br, THRESH_BR)
pure_br = res_br['new_topics']
print(f"  topics {len(set(orig_br[orig_br>=0]))} -> {len(set(pure_br[pure_br>=0]))} after purge "
      f"| reassigned {res_br['q_idx'].size:,} | unassigned {100*(pure_br==-1).mean():.2f}%")

br['topic_orig'] = orig_br
br['topic']      = pure_br
feather.write_feather(br, cfg_br['out'] / 'chunk_topics.feather')
save_spread_table(res_br['spread'], res_br['masked'], orig_br, THRESH_BR, cfg_br['out'])
save_codebook_spread(tm_br, pure_br, res_br['spread'], cfg_br['out'])
print('BR purified -> chunk_topics.feather (topic=purified, topic_orig=pre-purge) + codebook + spread table')
toc('br', '7 purify + kNN')

tic()
# BR — (validation 1) project the 200 OUT-OF-SAMPLE docs, export for manual reading
Xoos     = np.asarray(emb_br[oos_mask]).astype('float32')
docs_oos = br.loc[oos_mask, 'chunk_text'].tolist()
oos_topics = assign_oos(tm_br, L2T_br, Xoos)
print(f"OOS projected: {len(oos_topics):,} chunks | {100*(oos_topics==-1).mean():.2f}% unassigned")

rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(docs_oos), size=min(N_READ, len(docs_oos)), replace=False)
export_for_reading(cfg_br['out'] / 'read_oos_projection.csv', idx, docs_oos, oos_topics, tm_br)

# BR — (validation 2) export a manual-reading sample of ASSIGNED (training) OUTLIERS
rng = np.random.default_rng(RANDOM_STATE)
out_pos = np.where(out_tr)[0]
idx = rng.choice(out_pos, size=min(N_READ, len(out_pos)), replace=False)
export_for_reading(cfg_br['out'] / 'read_assigned_outliers.csv', idx, docs_tr, final_tr, tm_br)

# BR — EXPORT topic documentation to Excel with BOTH pre- and post-purification info in one file.
# Run AFTER BR SPREAD (for kw_doc_spread) and ideally after BR PURIFY (for n_chunks_purified / topic_orig).
def _flat(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return ' | '.join(_flat(e) for e in v if str(e).strip() != '')
    return v
info = tm_br.get_topic_info().copy()
for c in info.columns:
    if info[c].map(lambda v: isinstance(v, (list, tuple, np.ndarray))).any():
        info[c] = info[c].map(_flat)

ft = feather.read_table(cfg_br['out'] / 'chunk_topics.feather').to_pandas()
has_orig = 'topic_orig' in ft.columns
oc = (ft['topic_orig'] if has_orig else ft['topic']).value_counts()
pc = ft['topic'].value_counts()
info['n_chunks_orig']     = info['Topic'].map(oc).fillna(0).astype(int)   # PRE-purification (full corpus)
info['n_chunks_purified'] = info['Topic'].map(pc).fillna(0).astype(int)   # POST-purification (0 => masked-away)

sp_path = cfg_br['out'] / 'topics_kw_doc_spread.csv'
if sp_path.exists():
    spdf = pd.read_csv(sp_path)[['topic', 'kw_doc_spread', 'masked']]      # spdf, NOT sp (sp = scipy.sparse!)
    info = info.merge(spdf, left_on='Topic', right_on='topic', how='left').drop(columns='topic')
else:
    print('  (topics_kw_doc_spread.csv not found — run BR SPREAD first for kw_doc_spread)')
info.to_excel(cfg_br['out'] / 'topic_info.xlsx', index=False)
print('saved topic_info.xlsx |', len(info), 'topics | cols:', list(info.columns))

# BR — VERIFY the exported Excel actually has keywords + OpenAI label + exemplar docs
chk = pd.read_excel(cfg_br['out'] / 'topic_info.xlsx')
print('columns:', list(chk.columns))
need = ['Topic', 'Count', 'Representation', 'OpenAI', 'Representative_Docs']
print('MISSING:', [c for c in need if c not in chk.columns])
for c in [x for x in ['Representation', 'OpenAI', 'Representative_Docs'] if x in chk.columns]:
    empty = chk[c].isna() | (chk[c].astype(str).str.strip().isin(['', 'nan']))
    print(f'  {c}: {int(empty.sum())}/{len(chk)} empty')
with pd.option_context('display.max_colwidth', 100, 'display.width', 200):
    print(chk.head(3).to_string())
toc('br', '8 OOS + exports')


===== BR: purify + reads + topic_info =====
BR spread threshold = 20.5 (keep kw_doc_spread > 20) | total docs 16,830
purify: thresh=20.5 | 0/168 topics masked (kw_doc_spread<20.5)
  kNN: nothing masked -> no reassignment
  topics 168 -> 168 after purge | reassigned 0 | unassigned 0.00%
spread table -> topics_kw_doc_spread.csv | masked 0
codebook: 168 purified topics -> topics_codebook.csv
BR purified -> chunk_topics.feather (topic=purified, topic_orig=pre-purge) + codebook + spread table
  [time] br 7 purify + kNN: 46.6s


project: 100%|██████████| 1/1 [00:03<00:00,  3.04s/blk]


OOS projected: 33,630 chunks | 0.00% unassigned
wrote read_oos_projection.csv ( 200 rows for manual reading )
wrote read_assigned_outliers.csv ( 200 rows for manual reading )
saved topic_info.xlsx | 168 topics | cols: ['Topic', 'Count', 'Name', 'Representation', 'KeyBERT', 'Representative_Docs', 'n_chunks_orig', 'n_chunks_purified', 'kw_doc_spread', 'masked']
columns: ['Topic', 'Count', 'Name', 'Representation', 'KeyBERT', 'Representative_Docs', 'n_chunks_orig', 'n_chunks_purified', 'kw_doc_spread', 'masked']
MISSING: ['OpenAI']
  Representation: 6/168 empty
  Representative_Docs: 0/168 empty
   Topic  Count                                      Name                                                                                                     Representation                                                                                                                    KeyBERT                                                                                                         

In [8]:
print('===== before/after keyword export (both corpora) =====')
tic()
print('=== US ==='); us_ba = export_before_after(cfg_us['out'], cfg_us['lang'])
toc('us', '7 before/after export')
tic()
print('=== BR ==='); br_ba = export_before_after(cfg_br['out'], cfg_br['lang'])
toc('br', '9 before/after export')

# --- DONE marker: lets the local side confirm the whole refit finished ---
import time as _t
(ROOT/'scratch').mkdir(parents=True, exist_ok=True)
(ROOT/'scratch'/'_refit_done.txt').write_text('REFIT_DONE mcs=%d t=%.0f\n' % (MCS_FIXED, _t.time()))
print('ALL DONE -> scratch/_refit_done.txt')


===== before/after keyword export (both corpora) =====
=== US ===
saved topic_model_us/topic_info_before_after.xlsx | 101 topics | masked 6 | cols ['topic', 'n_chunks_before', 'n_chunks_after', 'masked', 'kw_doc_spread', 'keywords_before', 'keywords_after']
  [time] us 7 before/after export: 1.7s
=== BR ===
saved topic_model_br/topic_info_before_after.xlsx | 168 topics | masked 0 | cols ['topic', 'n_chunks_before', 'n_chunks_after', 'masked', 'kw_doc_spread', 'keywords_before', 'keywords_after']
  [time] br 9 before/after export: 10.7s
ALL DONE -> scratch/_refit_done.txt


In [9]:
# --- append this run to the pipeline timing register (code/timing.py is its only writer) ---
import sys
sys.path.insert(0, str(ROOT / 'code'))
from timing import log_run, CSV_PATH, MD_PATH

note = (f"{torch.cuda.get_device_name(0)}. BERTopic mcs={MCS_FIXED}, UMAP nn={N_NEIGHBORS} "
        f"d={N_COMPONENTS}, HDBSCAN ms={MIN_SAMPLES}, kNN reassign={KNN_REASSIGN}. "
        f"US fits all chunks; BR fits {N_PER_PARTY} docs/party and projects the rest.")
print(log_run('02 topic model', {c: t for c, t in TIMING.items() if t},
              scale={c: SCALE[c] for c in TIMING if TIMING[c]},
              started=RUN_STARTED, note=note))
print('->', CSV_PATH)
print('->', MD_PATH)


## 02 topic model · 2026-08-13 02:53:54 · 47d1065aba4b, 12 cores

NVIDIA A100-SXM4-80GB. BERTopic mcs=400, UMAP nn=60 d=10, HDBSCAN ms=25, kNN reassign=50. US fits all chunks; BR fits 150 docs/party and projects the rest.

| step | United States | Brazil |
|---|---:|---:|
| 1 load+align | 0:00:14 | 0:00:00 |
| 2 fit | 0:00:53 | 0:02:31 |
| 3 save model | 0:00:28 | 0:01:12 |
| 4 outlier assign + write | 0:00:08 | 0:00:07 |
| 5 purify + kNN | 0:00:33 | 0:00:00 |
| 6 exports | 0:00:02 | 0:00:00 |
| 7 before/after export | 0:00:02 | 0:00:00 |
| 1 load+align+filter+sample | 0:00:00 | 0:01:16 |
| 5 re-represent | 0:00:00 | 0:00:16 |
| 6 label all chunks | 0:00:00 | 0:05:55 |
| 7 purify + kNN | 0:00:00 | 0:00:47 |
| 8 OOS + exports | 0:00:00 | 0:00:11 |
| 9 before/after export | 0:00:00 | 0:00:11 |
| **total** | **0:02:19** | **0:12:26** |

| corpus | documents | chunks | seconds per 1k chunks |
|---|---:|---:|---:|
| United States | 4,507 | 439,282 | 0.3 |
| Brazil | 16,830 | 3,081,612 | 0.2

In [10]:
# --- VERIFY artifacts on Drive before disconnecting ---
from pathlib import Path
ROOT = Path(os.getenv('TOPIC2IRT_ROOT', '/content/drive/MyDrive/Papers/transfer_learning/topic2irt'))
ok = True
for corp in ['us', 'br']:
    d = ROOT / f'data/topics/topic_model_{corp}'
    print(f'=== {corp} : {d} ===')
    need = ['bertopic_model', 'chunk_topics.feather', 'topic_info.xlsx',
            'topic_info_before_after.xlsx', 'topics_codebook.csv', 'topics_kw_doc_spread.csv']
    for f in need:
        p = d / f
        if p.exists():
            sz = p.stat().st_size if p.is_file() else sum(x.stat().st_size for x in p.rglob('*') if x.is_file())
            print(f'  OK   {f:34s} {sz/1e6:8.2f} MB')
        else:
            print(f'  !!!  MISSING {f}'); ok = False
print('sentinel:', (ROOT / 'scratch' / '_refit_done.txt').read_text().strip())
print('ALL ARTIFACTS PRESENT' if ok else 'SOMETHING MISSING — do NOT disconnect')


=== us : /content/drive/MyDrive/Papers/transfer_learning/topic2irt/data/topics/topic_model_us ===
  OK   bertopic_model                      1343.53 MB
  OK   chunk_topics.feather                  39.40 MB
  OK   topic_info.xlsx                        0.03 MB
  OK   topic_info_before_after.xlsx           0.02 MB
  OK   topics_codebook.csv                    0.02 MB
  OK   topics_kw_doc_spread.csv               0.00 MB
=== br : /content/drive/MyDrive/Papers/transfer_learning/topic2irt/data/topics/topic_model_br ===
  OK   bertopic_model                      2825.91 MB
  OK   chunk_topics.feather                 294.05 MB
  OK   topic_info.xlsx                        0.04 MB
  OK   topic_info_before_after.xlsx           0.02 MB
  OK   topics_codebook.csv                    0.04 MB
  OK   topics_kw_doc_spread.csv               0.00 MB
sentinel: REFIT_DONE mcs=400 t=1786590631
ALL ARTIFACTS PRESENT


In [11]:
# --- DISCONNECT: flush Drive writes, then release the GPU runtime to stop credit usage ---
print('flushing Drive + releasing GPU runtime...')
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Drive flushed + unmounted')
except Exception as e:
    print('flush warn:', e)
from google.colab import runtime
runtime.unassign()   # releases the GPU-backed runtime -> stops burning credits
print('runtime released')


flushing Drive + releasing GPU runtime...
Drive flushed + unmounted
runtime released
